In [ ]:
import numpy as np
from scipy.optimize import minimize
from scipy.stats import qmc
import pandas as pd
import time


In [ ]:
class TunnelingAlgorithm:
    def __init__(self, f, f_grad, bounds, verbose=False):
        self.f = f # Objective function
        self.f_grad = f_grad # Gradient of the objective function
        self.bounds = bounds # Bounds for the optimization variables; list of (lower, upper) tuples
        self.dim = len(bounds) # Dimensionality of the problem
        self.x_stars = [] # Stores all minimizers at the current f_star level
        self.lambda_list = [] # Pole strengths for each x_star
        self.f_star = np.inf # Best known minimum value of f

        self.eps1 = 1e-9 # Threshold for considering a new minimum better than f_star
        self.eps2 = 1e-5 # Transition width for the ramp function in eta
        self.eps3 = 1e-3 # Threshold for considering a point as a good enough to end the tunneling phase
        self.lambda_max = 5 # Maximum pole strength to consider during lambda search
        self.ns = 100 # Max iterations for the restoration algorithm during the tunneling phase
        self.nb = 20 # Max bisections for step-size
        self.n_eps_attempts = 2 * self.dim # Number of attempts starting at the previous minimum in the tunneling phase
        self.n_random_starts = 2 * self.dim # Number of random starts when local perturbations fail in the tunneling phase
        self.max_cycles = 1000 # Maximum number of cycles of minimization and tunneling phases to perform

        self.verbose = verbose # If True, print detailed logs during execution
        self.minimization_phase_counter = 0 # Counter to track how many times the minimization phase is called
        self.success = 0 # Counter for successful steps in the restoration algorithm
        self.failure = 0 # Counter for failed steps in the restoration algorithm

    '''
    Input: Initial point `x_0` for the tunneling algorithm, and maximum number of cycles to perform.
    Output: A tuple containing the list of minimizers `x_stars` found at the best known minimum level `f_star`.

    The algorithm alternates between a minimization phase and a tunneling phase.
    '''
    def apply_algorithm(self, x_0):
        current_x = x_0
        for _ in range(self.max_cycles):
            # 1. Minimization Phase
            print(f"starting minimization phase at {current_x} with f={self.f(current_x)}") if self.verbose else None
            x_star, f_val = self.minimization_phase(current_x)
            print(f"Minimization phase completed at {x_star} with f={f_val}") if self.verbose else None

            
            # If a new unique global minimizer was found, update global minimum and manage poles
            if f_val < self.f_star - self.eps1:
                print(f"New unique global minimizer found at {x_star} with f={f_val}") if self.verbose else None
                self.f_star = f_val
                self.x_stars = [x_star]
                self.lambda_list = [1.0] # Reset to 1.0 for the new minimum

            # If a new minimizer at the same level is found, add it to the list of minimizers
            elif abs(f_val - self.f_star) <= self.eps1:
                # (Double-check to avoid duplicates)
                if not any(np.linalg.norm(x_star - prev) < 1e-3 for prev in self.x_stars):
                    print(f"New minimizer found at {x_star} with f={f_val}") if self.verbose else None
                    self.x_stars.append(x_star)
                    self.lambda_list.append(1.0) # Add a new pole for the new minimum
                else:
                    print(f"Duplicate minimizer found at {x_star} with f={f_val}, ignoring.") if self.verbose else None
            else:
                print(f"No improvement found at {x_star} with f={f_val}, current best is {self.f_star} at {self.x_stars}") if self.verbose else None
            
            # 2. Tunneling Phase
            print(f"Starting tunneling phase from {x_star} with f={f_val}") if self.verbose else None
            next_start = self.tunneling_phase(x_star)
            print(f"Tunneling phase completed, next start point: {next_start}") if self.verbose else None
            
            if next_start is None:
                break 
            current_x = next_start
            
        return self.x_stars, self.f_star

    '''
    Input: Initial point `x_0` for the minimization phase.
    Output: A tuple containing the minimizer `x_star` and its function value `f_val`.

    The library function `scipy.optimize.minimize` is used for the minimization phase, 
    and any local optimization method that supports bounds can be selected as the method.  
    '''
    def minimization_phase(self, x_0):
        self.minimization_phase_counter += 1
        res = minimize(self.f, x_0, jac=self.f_grad, bounds=self.bounds, method='L-BFGS-B')
        return res.x, res.fun


    '''
    Input: The last found local minimizer `x_star`.
    Output: A new starting point for the next minimization phase, or `None` if no valid point is found.

    The tunneling phase first attempts local perturbations around the last found minimizer to escape its basin of attraction.
    If local perturbations fail, it performs a global search by randomly sampling points within the bounds 
        and applying the restoration algorithm.
    '''
    def tunneling_phase(self, x_star):
        # 1. Local search: Try random perturbations around the last found minimizer
        for nr_eps in range(self.n_random_starts):
            epsilon = np.random.uniform(-0.1, 0.1, self.dim)
            while np.allclose(x_star, self.apply_bounds(x_star + epsilon)):
                epsilon = np.random.uniform(-0.1, 0.1, self.dim)
            print(f"Attempting local perturbation {nr_eps} around {x_star} with epsilon={epsilon}") if self.verbose else None

            # Determine the pole strength in the tunneling function for this minimizer:
            print(f"looking for appropriate lambda for minimizer") if self.verbose else None
            self.lambda_list[-1] = self.find_iterative_lambda(x_star, epsilon)
            print(f"Found lambda={self.lambda_list[-1]} for minimizer {x_star}") if self.verbose else None

            # Search for a point x with T(x) <= eps3, starting the search from x_star + epsilon:
            print(f"Running restoration algorithm from {x_star + epsilon}") if self.verbose else None
            next_start = self.run_restoration(x_star + epsilon)
            if next_start is not None:
                print(f"Found valid start point at {next_start}") if self.verbose else None
                return next_start
            else:
                print(f"Local perturbation {nr_eps} around {x_star} failed to find a valid start point.") if self.verbose else None

        # 2. Global search: If local search fails, try random points in Omega
        print(f"Global search initiated for minimizer {x_star}") if self.verbose else None
        return self.random_tunnel_search()

    '''
    Input: A new minimizer `x_star` and and a perturbation vector `epsilon_vec`.
    Output: A pole strength `lambda` for the pole at `x_star` in the tunneling function.

    This function iteratively tests increasing values of `lambda` for the pole being added at `x_star` to ensure that the 
        tunneling function's gradient points in a direction that allows escape from the current basin of attraction.

    The descent check is an implementation of condition (A.2) from the paper, checking that `delta_x` is a descent direction.
    The outward check is an implementation of condition (A.2a) from the paper, checking that `delta_x` points away from the pole. 
    
    '''
    def find_iterative_lambda(self, x_star, epsilon):
        l_trial = 1.0
        x_test = x_star + epsilon
        while l_trial <= self.lambda_max:
            # Temporarily apply l_trial
            self.lambda_list[-1] = l_trial
            
            # Get the displacement vector delta_x (from section 2.3.4)
            delta_x = self.get_displacement(x_test, x_star, 0.0)
            t_grad = self.get_t_and_grad(x_test, x_star, 0.0)[1]

            descent_check = np.dot(t_grad, delta_x) < 0
            outward_check = np.dot(epsilon, delta_x) > 0
            
            if descent_check and outward_check:
                return l_trial
                    
            l_trial += 1
        
        return self.lambda_max

    '''
    Input: An iterate `x_start` that serves as the initial point for the restoration algorithm.
    Output: A new starting point for the next minimization phase, or `None` if no valid point is found.

    This function implements the restoration algorithm as described in section 2.3.4 of the paper, which is used 
        during the tunneling phase to find a new point that satisfies T(x) < eps3.
    
    '''
    def run_restoration(self, x_start):
        x_hat = self.apply_bounds(x_start) # The point that we move away from
        x_m = np.copy(x_start) # Initial movable pole position
        lambda_0 = 0.0         # Initial movable pole strength
        
        for _ in range(self.ns):
            t_val, t_grad = self.get_t_and_grad(x_hat, x_m, lambda_0)
            
            # 1. Check for Tunnel Exit
            if t_val <= self.eps3:
                # Must be far from ALL found minima to be a valid new start
                if all(np.linalg.norm(x_hat - prev) > 1e-2 for prev in self.x_stars):
                    return x_hat

            # 2. Calculate Displacement (Section 2.3.4)
            delta_x = self.get_displacement(x_hat, x_m, lambda_0)

            # 3. Step-Size Bisection
            alpha = 1.0
            x_new = x_hat # Default
            success = False
            for _ in range(self.nb):
                x_try = self.apply_bounds(x_hat + alpha * delta_x)
                if self.get_t_and_grad(x_try, x_m, lambda_0)[0] < t_val:
                    print(f"Next iterate of restoration algorithm: {x_try} with T={self.get_t_and_grad(x_try, x_m, lambda_0)[0]}. Previous t_val was {t_val}. alpha = {alpha} and delta_x = {delta_x}") if self.verbose else None
                    x_new = x_try
                    success = True
                    self.success += 1
                    break
                alpha /= 2.0
            
            if not success:
                print(f"Restoration algorithm failed to find a better point from {x_hat} with T={t_val}") if self.verbose else None
                self.failure += 1

            # 4. Handle Movable Pole (Appendix I, Step 3)
            x_m = self.determine_xm(x_hat, x_new)
            print(f"Updated movable pole position to x_m={x_m}") if self.verbose else None
            delta_x_from_x_new_to_x_tilde = self.get_displacement(x_new, x_m, lambda_0)
            u = np.dot(x_new - x_hat, delta_x_from_x_new_to_x_tilde) 
            if u <= 0:
                # landed in undesirable local minimum of T(x); update pole[cite: 1]
                lambda_0 = self.find_iterative_lambda_0(x_hat, x_new, x_m)
                print(f"Updating lambda_0 to {lambda_0} for movable pole at x_m={x_m}") if self.verbose else None
            else:
                # Heuristic reset rule (A.9): return to simpler geometry[cite: 1]
                # Compute displacement for lambda_0 = 0[cite: 1]
                delta_x_0 = self.get_displacement(x_new, x_m, 0.0)
                if np.dot(delta_x_0, delta_x_from_x_new_to_x_tilde) > 0:
                    print(f"Heuristic reset: resetting lambda_0 to 0 for movable pole at x_m={x_m}") if self.verbose else None
                    lambda_0 = 0.0

            x_hat = x_new
            
        return None
    
    '''
    Input: An iterate `x_hat`, the following iterate `x_new`, and the movable pole position `x_m`.
    Output: A pole strength `lambda_0` for the movable pole in the tunneling function.

    This function iteratively tests increasing values of `lambda_0` for the movable pole to ensure that the
        displacement vector from section 2.3.4 points in a direction that allows escape from the current basin of attraction.
    
    '''
    def find_iterative_lambda_0(self, x_hat, x_new, x_m):
        l0 = 1.0
        while l0 <= self.lambda_max:
            delta_x_from_x_new_to_x_tilde = self.get_displacement(x_new, x_m, l0)
            # If the step taken away from x_new is in the same general direction (acute angle) as the step taken from 
            # x_hat to x_new, then the lambda_0 is satisfactory. Otherwise, increase lambda_0 and try again.
            if np.dot(x_new - x_hat, delta_x_from_x_new_to_x_tilde) > 0:
                print(f"Found lambda_0={l0} for movable pole at x_m={x_m}") if self.verbose else None
                return l0
            l0 += 1.0
        print(f"Warning: lambda_0 reached maximum value of {self.lambda_max} without satisfying condition.") if self.verbose else None
        return self.lambda_max


    '''
    Input: An iterate `x`, the movable pole position `x_m`, and the movable pole strength `lambda_0`.
    Output: The displacement vector for the restoration algorithm in the tunneling phase

    This function calculates the displacement vector as described in section 2.3.4 of the paper, which 
        is used to guide the restoration algorithm during the tunneling phase.
    '''
    def get_displacement(self, x, x_m, lambda_0):
        t_val, t_grad = self.get_t_and_grad(x, x_m, lambda_0)
        denom = np.dot(t_grad, t_grad)
        # Eq. in section 2.3.4[cite: 1]
        print(f"Calculating displacement with t_val={t_val}, t_grad={t_grad}, and denom={denom}") if self.verbose else None
        return -(t_val / max(denom, 1e-20)) * t_grad

    '''
    Input: Two iterates `x_hat` (an iterate from the tunneling phase) and `x_new` (the following iterate).
    Output: A new position `x_m` for the movable pole.

    This function determines the new position for the movable pole when the tunneling algorithm lands in an undesirable 
        local minimum of the tunneling function.

    The calculation of `x_m` is based on equation (A.7) from the paper, with xi = 0.9 * dist.
    '''
    def determine_xm(self, x_hat, x_new):

        if np.linalg.norm(x_hat - x_new) != 0:
            return x_new + (x_hat - x_new) * 0.5 / np.linalg.norm(x_hat - x_new) #here222
        else:
            return x_hat

        dist = np.linalg.norm(x_hat - x_new)
        if dist < 1: return np.copy(x_hat)
        xi = 0.9 / dist # 1.5 seemed a little better in a small test
        print(f"Updating movable pole position to xm={xi*x_hat + (1 - xi) * x_new}") if self.verbose else None
        return xi * x_hat + (1 - xi) * x_new

    '''
    Input: An iterate `x`, the movable pole position `x_m`, and the movable pole strength `lambda_0`.
    Output: The value of the tunneling function T(x) and its gradient at `x`.

    This function computes the tunneling function T(x) and its gradient based on the current known minima (fixed poles), which are
        stored as `self.x_stars`, the corresponding strengths stored in `self.lambda_list`, and the movable pole with its strength.
    
    The function T(x) is defined as equation (A.0) in the paper, and the gradient is derived using the product and chain rules.
    '''
    def get_t_and_grad(self, x, x_m, lambda_0):
        f_val = self.f(x)
        f_diff = f_val - self.f_star
        grad_f = self.f_grad(x)
        
        denom = 1.0
        # This vector will store grad(D)/D
        grad_D_over_D = np.zeros_like(x)
        
        # 1. Process Fixed Poles
        for i, x_star_i in enumerate(self.x_stars):
            diff = x - x_star_i
            dist = np.linalg.norm(diff)
            dist_sq = max(dist**2, 1e-20)
            
            # Calculate eta and its derivative (Ramp Logic from equation (A.5) in the paper)
            if dist >= 1 + self.eps2:
                eta = 0.0
                grad_eta = np.zeros_like(x)
            elif dist <= 1 - self.eps2:
                eta = self.lambda_list[i]
                grad_eta = np.zeros_like(x)
            else:
                # Transition zone
                eta = (self.lambda_list[i] / 2) * (1 + (1 - dist) / self.eps2)
                grad_eta = -(self.lambda_list[i] / (2 * self.eps2)) * (diff / (dist + 1e-20))
            
            denom *= (dist_sq ** eta)
            
            # d/dx [eta(x) * ln(dist_sq)] = grad_eta * ln(dist_sq) + eta * (2*diff / dist_sq)
            grad_D_over_D += grad_eta * np.log(dist_sq) + (eta * 2 * diff) / dist_sq

        # 2. Process Movable Pole (lambda_0 is constant during the step)
        diff_m = x - x_m
        dist_sq_m = max(np.sum(diff_m**2), 1e-20)
        denom *= (dist_sq_m ** lambda_0)
        grad_D_over_D += (lambda_0 * 2 * diff_m) / dist_sq_m

        # 3. Final T and grad(T)
        t_val = f_diff / max(denom, 1e-20)
        # grad(T) = (grad_f / denom) - T * (grad_D / D)
        t_grad = (grad_f / max(denom, 1e-20)) - (t_val * grad_D_over_D)
        
        return t_val, t_grad

    '''
    Input: An iterate `x` that may or may not be within the bounds.
    Output: The iterate `x` clamped to the bounds.

    This function ensures that the iterate `x` remains within the specified bounds.
    '''
    def apply_bounds(self, x):
        return np.clip(x, [b[0] for b in self.bounds], [b[1] for b in self.bounds])


    '''
    Input: None (uses class attributes for bounds and dimensions).
    Output: A new starting point for the next minimization phase, or `None` if no valid point is found.

    This function implements the global search strategy of the tunneling phase by randomly sampling points within the bounds 
        and applying the restoration algorithm to find a valid starting point for the next minimization phase.
    It is used when local perturbations around the last found minimizer fail to find a new valid minimizer.
    
    '''
    def random_tunnel_search(self):
        
        for i_random_start in range(self.n_random_starts):
            # Generate a random point within the bounds
            x_rand = np.array([np.random.uniform(b[0], b[1]) for b in self.bounds])
            print(f"Global search: trying random point {x_rand} (attempt {i_random_start + 1})") if self.verbose else None
            # Initiate the restoration algorithm from this random point[cite: 1]
            # For random starts, the paper assumes lambda_0 = 0[cite: 1]
            res = self.run_restoration(x_rand)
            print(f"Global search: restoration algorithm returned {res} for random point {x_rand}") if self.verbose else None
            if res is not None:
                return res
                
        return None



In [ ]:
class MRSAlgorithm():
    def __init__(self, f, f_grad, bounds, eps1=1e-9):
        self.f = f # Objective function
        self.f_grad = f_grad # Gradient of the objective function
        self.bounds = bounds # Bounds for the optimization variables; list of (lower, upper) tuples
        self.dim = len(bounds) # Dimensionality of the problem
        self.x_stars = [] # Stores all minimizers at the current f_star level
        self.f_star = np.inf # Best known minimum value of f
        self.eps1 = eps1 # Threshold for considering a new minimum better than f_star

        self.minimization_phase_counter = 0 # Counter to track how many times the minimization phase is called

    def apply_algorithm(self, x_0):
        max_cycles = x_0.shape[0] * 1000 # 1000 cycles per dimension
        for _ in range(max_cycles):
            x_0 = np.random.uniform([b[0] for b in self.bounds], [b[1] for b in self.bounds])
            res = minimize(self.f, x_0, jac=self.f_grad, bounds=self.bounds, method='L-BFGS-B')
            self.minimization_phase_counter += 1
            if res.fun < self.f_star - self.eps1:
                self.f_star = res.fun
                self.x_stars = [res.x]
            elif abs(res.fun - self.f_star) <= self.eps1:
                if not any(np.linalg.norm(res.x - prev) < 1e-3 for prev in self.x_stars):
                    self.x_stars.append(res.x)
        return self.x_stars, self.f_star

            

In [ ]:
"""
Shubert function
"""
def shubert(x):
    x = np.asarray(x)
    i = np.array([1, 2, 3, 4, 5])
    
    # Create a grid for broadcasting: (n, 1) and (1, 5)
    # This computes all i-terms for all x_j simultaneously
    inner_terms = i * np.cos((i + 1) * x[:, np.newaxis] + i)
    
    # Sum across the i-dimension (axis 1) then product across the x-dimension
    s_j = np.sum(inner_terms, axis=1)
    return np.prod(s_j)

def shubert_grad(x):
    x = np.asarray(x)
    n = len(x)
    i = np.arange(1, 6)
    
    # 1. Compute S_j (sums) and dS_j (derivatives of sums)
    # Shapes: x is (n, 1), i is (1, 5) -> result is (n, 5)
    x_grid = x[:, np.newaxis]
    
    s_terms = i * np.cos((i + 1) * x_grid + i)
    ds_terms = -i * (i + 1) * np.sin((i + 1) * x_grid + i)
    
    s = np.sum(s_terms, axis=1)   # Shape (n,)
    ds = np.sum(ds_terms, axis=1) # Shape (n,)
    
    # 2. Compute "product of all others" using prefix and suffix products
    # This avoids O(n^2) operations or division by zero errors.
    prefix = np.ones(n + 1)
    suffix = np.ones(n + 1)
    
    prefix[1:] = np.cumprod(s)
    suffix[:-1] = np.cumprod(s[::-1])[::-1]
    
    # The product excluding index j is prefix[j] * suffix[j+1]
    grad_others = prefix[:-1] * suffix[1:]
    
    return ds * grad_others



"""
Altered Shubert function
"""
def altered_shubert(x):
    return shubert(x) + 0.5 * ((x[0] + 1.42513) ** 2 + (x[1] + 0.80032) ** 2)

def altered_shubert_grad(x):
    grad_shubert = shubert_grad(x)
    grad_alteration = np.array([x[0] + 1.42513, x[1] + 0.80032])
    return grad_shubert + grad_alteration



"""
Six-Hump Camel function
"""
def six_hump_camel(x):
    x1, x2 = x
    term1 = (4 - 2.1 * x1**2 + (x1**4) / 3) * x1**2
    term2 = x1 * x2
    term3 = (-4 + 4 * x2**2) * x2**2
    return term1 + term2 + term3

def six_hump_camel_grad(x):
    x1, x2 = x
    # Gradient with respect to x1
    dterm1_dx1 = (4 - 2.1 * x1**2 + (x1**4) / 3) * 2 * x1 + (4/3 * x1**3 - 4.2 * x1) * x1**2
    dterm2_dx1 = x2
    dterm3_dx1 = 0

    # Gradient with respect to x2
    dterm1_dx2 = 0
    dterm2_dx2 = x1
    dterm3_dx2 = (-4 + 4 * x2**2) * 2 * x2 + (8 * x2) * x2**2

    return np.array([dterm1_dx1 + dterm2_dx1 + dterm3_dx1, dterm1_dx2 + dterm2_dx2 + dterm3_dx2])


"""
Function 16
"""
def function_16(x, k=10, A=1):
    """Implementation of Equation (16) from image_1d2eb3.png."""
    x = np.asarray(x)
    n = len(x)
    pi = np.pi
    
    term1 = k * (np.sin(pi * x[0])**2)
    sum_part = np.sum((x[:-1] - A)**2 * (1 + k * np.sin(pi * x[1:])**2))
    term3 = (x[-1] - A)**2
    
    return (pi / n) * (term1 + sum_part + term3)

def function_16_grad(x, k=10, A=1):
    """Gradient of Equation (16) from image_1d2eb3.png."""
    x = np.asarray(x)
    n = len(x)
    pi = np.pi
    
    grad = np.zeros_like(x, dtype=float)
    s = np.sin(pi * x)
    s2 = np.sin(2 * pi * x) # derivative of sin^2(pi*x) is pi*sin(2*pi*x)

    # Initial terms
    grad[0] += k * pi * s2[0]
    grad[-1] += 2 * (x[-1] - A)

    if n > 1:
        # Derivative of (x_i - A)^2 * (1 + k*sin^2(pi*x_{i+1})) wrt x_i
        grad[:-1] += 2 * (x[:-1] - A) * (1 + k * s[1:]**2)
        # Derivative of (x_i - A)^2 * (1 + k*sin^2(pi*x_{i+1})) wrt x_{i+1}
        grad[1:] += (x[:-1] - A)**2 * k * pi * s2[1:]

    return (pi / n) * grad

def function_15(x, k=10, A=1):
    x = np.asarray(x)
    y = 1 + 0.25 * (x - 1)
    return function_16(y, k, A)

def function_15_grad(x, k=10, A=1):
    x = np.asarray(x)
    y = 1 + 0.25 * (x - 1)
    return 0.25 * function_16_grad(y, k, A)


"""
Function 17
"""
def function_17(x, k0=1.0, k1=0.1, A=1.0, l0=3.0, l1=2.0):
    """Implementation of Equation (17) from image_1cbd61.png."""
    x = np.asarray(x)
    n = len(x)
    pi = np.pi
    
    # Initial term
    term1 = k1 * (np.sin(pi * l0 * x[0])**2)
    
    # Summation term: i=1 to n-1
    sum_part = 0
    if n > 1:
        sum_part = np.sum((x[:-1] - A)**2 * (1 + k0 * np.sin(pi * l0 * x[1:])**2))
    
    # Final term (using l1 for the terminal frequency, which is not done in the paper, where l1 is defined but never used)
    term_final = (x[-1] - A)**2 * (1 + k0 * np.sin(pi * l1 * x[-1])**2)
    
    return term1 + k1 * sum_part + k1 * term_final

def function_17_grad(x, k0=1.0, k1=0.1, A=1.0, l0=3.0, l1=2.0):
    """Gradient of Equation (17) from image_1cbd61.png."""
    x = np.asarray(x)
    n = len(x)
    pi = np.pi
    grad = np.zeros_like(x, dtype=float)
    
    # Periodic terms
    s_l0 = np.sin(pi * l0 * x)
    s2_l0 = np.sin(2 * pi * l0 * x)
    s_l1 = np.sin(pi * l1 * x)
    s2_l1 = np.sin(2 * pi * l1 * x)
    
    # Coordinate 1
    grad[0] += pi * l0 * s2_l0[0]
    if n > 1:
        grad[0] += 2 * (x[0] - A) * (1 + k0 * s_l0[1]**2)
    
    # Coordinates 2 to n-1
    if n > 2:
        # Derivative wrt (x_i - A)^2
        grad[1:-1] += 2 * (x[1:-1] - A) * (1 + k0 * s_l0[2:]**2)
        # Derivative wrt sin^2(pi * l0 * x_i) from summation
        grad[1:-1] += (x[:-2] - A)**2 * k0 * pi * l0 * s2_l0[1:-1]
        
    # Coordinate n
    if n > 1:
        # From summation part
        grad[-1] += (x[-2] - A)**2 * k0 * pi * l0 * s2_l0[-1]
        
    # From final standalone term
    grad[-1] += 2 * (x[-1] - A) * (1 + k0 * s_l1[-1]**2)
    grad[-1] += (x[-1] - A)**2 * k0 * pi * l1 * s2_l1[-1]
    
    return k1 * grad


def altered_cos(x):
    x = np.asarray(x)
    return np.array([np.cos(x[0]) + 0.2 * x[0]**2])

def altered_cos_grad(x):
    x = np.asarray(x)
    return np.array([-np.sin(x[0]) + 0.4 * x[0]])

In [ ]:
def test_algorithm(algorithm, f, f_grad, bounds, global_min, x0_list):
    np.random.seed(42)
    nr_failures = 0
    found_minima_count = []
    for x0 in x0_list:
        algo = algorithm(f, f_grad, bounds)
        minima, f_min = algo.apply_algorithm(x0)
        if abs(f_min - global_min) < 1e-5:
            found_minima_count.append(len(minima))
        else:
            nr_failures += 1
        #print(f"Initial: {x0}, Found Min: {f_min:.6f}, Amount of Minima: {len(minima)}, Amount of Minimization Phases: {algo.minimization_phase_counter}")
    return np.mean(found_minima_count), nr_failures

In [ ]:
class Algorithm_Tester:
    def __init__(self):
        np.random.seed(42)
        shubert_2d_starting_points = np.random.uniform(-10, 10, (30, 2))
        shubert_3d_starting_points = np.random.uniform(-10, 10, (10, 3))
        shubert_6d_starting_points = np.random.uniform(-10, 10, (10, 6))
        six_camel_hump_starting_points = np.random.uniform(low=[-3, -2], high=[3, 2], size=(30, 2))
        altered_shubert_starting_points = np.random.uniform(-10, 10, (30, 2))
        function_15_2d_starting_points = np.random.uniform(-10, 10, (30, 2))
        function_15_3d_starting_points = np.random.uniform(-10, 10, (20, 3))
        function_15_4d_starting_points = np.random.uniform(-10, 10, (10, 4))
        function_16_5d_starting_points = np.random.uniform(-10, 10, (10, 5))
        function_16_8d_starting_points = np.random.uniform(-10, 10, (10, 8))
        function_16_10d_starting_points = np.random.uniform(-10, 10, (10, 10))


        self.test_list = []
        self.test_list.append({
            'name': 'Shubert 2D',
            'function': shubert, 
            'grad': shubert_grad, 
            'bounds': [[-10, 10], [-10, 10]], 
            'starting_points': shubert_2d_starting_points, 
            'global_min': -186.730909, 
            'minima_count': 18,
            'nr_trials': shubert_2d_starting_points.shape[0],
        })
        self.test_list.append({
            'name': 'Shubert 3D',
            'function': shubert, 
            'grad': shubert_grad, 
            'bounds': [[-10, 10], [-10, 10], [-10, 10]], 
            'starting_points': shubert_3d_starting_points, 
            'global_min': -2709.093506, 
            'minima_count': 81,
            'nr_trials': shubert_3d_starting_points.shape[0],
        })
        self.test_list.append({
            'name': 'Shubert 6D',
            'function': shubert, 
            'grad': shubert_grad, 
            'bounds': [[-10, 10]]*6, 
            'starting_points': shubert_6d_starting_points, 
            'global_min': -8272701.3783975, 
            'minima_count': 4374,
            'nr_trials': shubert_6d_starting_points.shape[0],
        })
        self.test_list.append({
            'name': 'Six Hump Camel',
            'function': six_hump_camel, 
            'grad': six_hump_camel_grad, 
            'bounds': [[-3, 3], [-2, 2]], 
            'starting_points': six_camel_hump_starting_points, 
            'global_min': -1.031628, 
            'minima_count': 2,
            'nr_trials': six_camel_hump_starting_points.shape[0],
        })
        self.test_list.append({
            'name': 'Altered Shubert 2D',
            'function': altered_shubert, 
            'grad': altered_shubert_grad, 
            'bounds': [[-10, 10], [-10, 10]], 
            'starting_points': altered_shubert_starting_points, 
            'global_min': -186.730909, 
            'minima_count': 1,
            'nr_trials': altered_shubert_starting_points.shape[0],
        })
        self.test_list.append({
            'name': 'Function 15 2D',
            'function': function_15, 
            'grad': function_15_grad,
            'bounds': [[-10, 10], [-10, 10]],
            'starting_points': function_15_2d_starting_points,
            'global_min': 0.0,
            'minima_count': 1,
            'nr_trials': function_15_2d_starting_points.shape[0],
        })
        self.test_list.append({
            'name': 'Function 15 3D',
            'function': function_15, 
            'grad': function_15_grad,
            'bounds': [[-10, 10], [-10, 10], [-10, 10]],
            'starting_points': function_15_3d_starting_points,
            'global_min': 0.0,
            'minima_count': 1,
            'nr_trials': function_15_3d_starting_points.shape[0],
        })
        self.test_list.append({
            'name': 'Function 15 4D',
            'function': function_15, 
            'grad': function_15_grad,
            'bounds': [[-10, 10], [-10, 10], [-10, 10], [-10, 10]],
            'starting_points': function_15_4d_starting_points,
            'global_min': 0.0,
            'minima_count': 1,
            'nr_trials': function_15_4d_starting_points.shape[0],
        })
        self.test_list.append({
            'name': 'Function 16 5D',
            'function': function_16, 
            'grad': function_16_grad,
            'bounds': [[-10, 10]]*5,
            'starting_points': function_16_5d_starting_points,
            'global_min': 0.0,
            'minima_count': 1,
            'nr_trials': function_16_5d_starting_points.shape[0],
        })
        self.test_list.append({
            'name': 'Function 16 8D',
            'function': function_16, 
            'grad': function_16_grad,
            'bounds': [[-10, 10]]*8,
            'starting_points': function_16_8d_starting_points,
            'global_min': 0.0,
            'minima_count': 1,
            'nr_trials': function_16_8d_starting_points.shape[0],
        })
        self.test_list.append({
            'name': 'Function 16 10D',
            'function': function_16, 
            'grad': function_16_grad,
            'bounds': [[-10, 10]]*10,
            'starting_points': function_16_10d_starting_points,
            'global_min': 0.0,
            'minima_count': 1,
            'nr_trials': function_16_10d_starting_points.shape[0],
        })

        self.results_list = []

    
    def run_tests(self, algorithm):
        for test in self.test_list:
            start_time = time.perf_counter()
            avg_nr_minima_found, nr_failures = test_algorithm(algorithm, test['function'], test['grad'], bounds=test['bounds'], global_min=test['global_min'], x0_list=test['starting_points'])
            end_time = time.perf_counter()
            time_per_trial = (end_time - start_time) / test['nr_trials']
            print(f"Test: {test['name']}, Avg Minima Found: {avg_nr_minima_found:.2f} out of {test['minima_count']}, Failure Rate: {nr_failures/test['nr_trials']:.2%} over {test['nr_trials']} trials, Time per Trial: {time_per_trial:.3f} seconds.")

        self.results_list.append({
            'function': test['name'], 
            'avg_pct_minima_found': avg_nr_minima_found/test['nr_trials'], 
            'total_minima': test['minima_count'], 
            'failure_rate': nr_failures/test['nr_trials'],
            'nr_trials': test['nr_trials']
        })
        


In [ ]:
algorithm_tester = Algorithm_Tester()
algorithm_tester.run_tests(TunnelingAlgorithm)
algorithm_tester.run_tests(MRSAlgorithm)
print(algorithm_tester.results_list)

In [ ]:
ta = TunnelingAlgorithm(altered_cos, altered_cos_grad, [[-10, 10]], verbose=True)
ta.apply_algorithm(np.array([np.random.uniform(-10, 10)]))
